# Datamine PLOTTX Process Example

This notebook demonstrates how to configure and run the **`plottx`** process wrapper in `dmstudio`.

## Process Description

## PLOTTX

Generate a plot from a text file. This enables legends, text etc. to be added to any plot file.

The text file is normally set up by the [INPUTC](<inputc.md>) process. However, any file containing explicit alphanumeric fields may be used. All alphanumeric words from the file will be amalgamated together, irrespective of the field name. Any implicit or numeric fields will be ignored with a warning message.

Because a blank line is ignored in a text file, blank lines are simulated by use of a four character combination. This is **** by default; any line starting with **** in the first four characters will be plotted as a blank line. If required, this set of four characters may be redefined using the @**SPACE** parameter.

A box is plotted around the text by default; this may be suppressed by setting @**NOBOX** =1. The box is separated from the text lines by a margin which is 4 millimetres wide by default. This may be changed by setting the @**MARGIN** parameter to the required distance. The size of the box is determined by the longest line of text in the file, rounded up to the nearest word (4 characters).

Positioning of the text is carried out by use of the @**XSTART** and @**YSTART** parameters, which give the position on the plot, in millimetres, of the top left hand corner of the box around the text (whether this box is plotted or not) from the plot origin (bottom left hand corner). The first line of text is plotted a distance of margin size plus 1.3 character sizes below this point, and starting margin size to the right of this point; each subsequent line of text starts 1.3 character sizes below the last, leaving a gap of 23 character sizes between each line of text.

The angle of the text may be specified by use of the optional @**ANGLE** parameter. Rotation angles specified in degrees clockwise about the X axis; 0 is parallel to the axis, 90 is vertically down, and -90 is vertically upwards. The text is rotated around the point @**XSTART** , @**YSTART**.

Because the text is positioned by millimetres on the plot, part of the plot prototype used is the total plot area, defined by the fields **XPICRT** , **YPICTP** in the plot prototype. The data area is ignored, and therefore data area parameters such as @**XMIN** ,........@**YSCALE** are not needed.

The process tests that the point @**XSTART** ,@**YSTART** lies in the plot region; however the user must ensure that there is space for the full text within the region.

### Input Files:

* **in** (Undefined):
  Input text file.
  Required=Yes

* **proto** (Plot Prototype):
  Plot prototype file. Must contain the fields **X, Y, S1, S2** and **CODE** (numeric,
  explicit) and **XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE** (numeric, implicit).
  Required=Yes

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

### Parameters:

* **xstart**:
  X position, in millimetres, for start of text (top left hand corner of box).
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **ystart**:
  Y position, in millimetres, for start of text (top left hand corner of box).
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **angle**:
  Angle of text clockwise from the X axis (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **nobox**:

* **Options** (0: a box is plotted around the text, =1 a box is not plotted around the text):
  (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **margin**:
  The margin, in millimetres, between the text and the surrounding box (3).
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **space**:
  A word which if encountered as the first word in a text record is interpreted as a
  blank line. Must be between quotes; e.g '----'. Default is ' '.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **charsize**:
  Character size in millimetres (3).
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **append**:
  Plot append flag. If set to 1 then the new plot will be appended to the **PLOT** file,
  if it exists and is a valid plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('plottx')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute plottx
print("Running plottx...")
dm_cmd.plottx(
    in_i='t_assays',  # required
    proto_i='t_mod1',  # required
    plot_o='t_plottx_out',  # required
    xstart_p='required_val',  # required
    ystart_p='required_val',  # required
    # angle_p=0,  # optional
    # nobox_p=0,  # optional
    # margin_p=3,  # optional
    # space_p='optional',  # optional
    # charsize_p=3,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("plottx execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_plottx_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")